    NOTE : In this notebook, we will be using cov_type as HC3 to get accurate CI's and p-values, as proved in
    4.LinearRegression.ipynb (its residual-vs-fitted diagnostics show heteroskedastic errors). The response
    variable is raw ROI (returns per rupee); a full log-scale cross-check of every verdict — on the scale
    validated in 3.DistributionFitting.ipynb — closes this notebook.

    All tests are evaluated at the significance level α = 0.05 (two-sided); the 95% confidence intervals correspond to the same level.


## Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv('CleanedDataSet.csv')
df['Scale_Bin'] = pd.cut(df['Movie_Scale'], bins=range(0, 11)).cat.remove_unused_categories()

In [2]:
bools = ['Remake', 'Franchise', 'Holiday']
genres = [  'Genre_adult', 'Genre_animation', 'Genre_comedy', 'Genre_documentary',
            'Genre_drama', 'Genre_fantasy', 'Genre_horror', 'Genre_love_story',
            'Genre_masala', 'Genre_mythological', 'Genre_rom_com', 'Genre_suspense',
            'Genre_thriller']
cast_novelty = ['New_Actor', 'New_Director', 'New_Music_Director']

focus_cols = bools + genres + cast_novelty

Y = df['ROI']

## H1 : Festive Timing
**Does holiday slot survive a like-to-like scaled comparison**

- What does RAW comparison say?

In [3]:
X = df['Holiday']
X = sm.add_constant(X)
rawmodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel.params['Holiday'], rawmodel.pvalues['Holiday']
lo, hi = rawmodel.conf_int().loc['Holiday']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}(approx)   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Holiday coefficient : +0.2262  
p-value = 0.0000(approx)   
95% CI [+0.1195, +0.3329]


- What does a content-adjusted (scale not yet controlled) model say?

In [4]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel.params['Holiday'], ROImodel.pvalues['Holiday']
lo, hi = ROImodel.conf_int().loc['Holiday']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Holiday coefficient : +0.1351  
p-value = 0.0061   
95% CI [+0.0386, +0.2316]


- What does a scale adjustment alone model say?

In [5]:
X = "ROI ~ Holiday" + " + C(Scale_Bin)"
raw_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale.params['Holiday[T.True]'], raw_scale.pvalues['Holiday[T.True]']
lo, hi = raw_scale.conf_int().loc['Holiday[T.True]']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")


Holiday coefficient : +0.0671  
p-value = 0.1388   
95% CI [-0.0218, +0.1560]


- what does a full model considering everything say?

In [6]:
X = "ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale.params['Holiday[T.True]'], full_scale.pvalues['Holiday[T.True]']
lo, hi = full_scale.conf_int().loc['Holiday[T.True]']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Holiday coefficient : +0.0598  
p-value = 0.1802   
95% CI [-0.0277, +0.1474]


### Result: **H1 is rejected**

| Model | Holiday effect (ROI) | 95% CI | p-value |
|---|---|---|---|
| Raw (no controls) | *+0.2262* | [+0.1195, +0.3329] | < 0.0001 |
| Content-adjusted, scale not controlled | *+0.1351* | [+0.0386, +0.2316] | 0.006 |
| Scale-adjusted only | *+0.0671* | [-0.0218, +0.1560] | 0.139 |
| Scale + content (full) | *+0.0598* | [-0.0277, +0.1474] | 0.180 |


1. Though intially Holiday has an +0.23 effect on ROI, almost 40% of it is covered purely via other content adjustment and almost 70% of it is covered just purely via Scale-adjustment.

2. Once both the content and scale adjustments are completed, the effect of Holiday on ROI has dropped drastically to +0.06, thus telling us that it has minimal effect on ROI once the other elements are accounted.

3. The p-value is also resonably large enough > 0.1 , hence we can with confidently rule out hypothesis 1.

4. This ROI verdict is consistent with 3.DistributionFitting.ipynb (Step 8): at high scale, festival films still show a large log-revenue gap ((8,9] +0.21, shuffle p = 0.023; (9,10] +0.56, about 76% more revenue, shuffle p = 0.001). That is a revenue effect, not a returns effect - the festival premium concentrates exactly where productions are biggest, so the extra revenue does not become extra ROI. Timing buys reach, not returns.

#### *Verdict : Holiday Slot does not survive*

## H2: Remakes and Franchise
**Do the remake and franchise slots survive a like-to-like scaled comparison**

### 1. Franchise

- What does RAW comparison say? (Franchise)

In [7]:
X = df['Franchise']
X = sm.add_constant(X)
rawmodel_f = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel_f.params['Franchise'], rawmodel_f.pvalues['Franchise']
lo, hi = rawmodel_f.conf_int().loc['Franchise']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +1.3794  
p-value = 0.0000   
95% CI [+1.0414, +1.7173]


- What does a content-adjusted (scale not yet controlled) model say? (Franchise)

In [8]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel_f = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel_f.params['Franchise'], ROImodel_f.pvalues['Franchise']
lo, hi = ROImodel_f.conf_int().loc['Franchise']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +1.1665  
p-value = 0.0000   
95% CI [+0.8329, +1.5001]


- What does a scale adjustment alone model say? (Franchise)

In [9]:
X = "ROI ~ Franchise + C(Scale_Bin)"
raw_scale_f = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale_f.params['Franchise[T.True]'], raw_scale_f.pvalues['Franchise[T.True]']
lo, hi = raw_scale_f.conf_int().loc['Franchise[T.True]']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +0.6183  
p-value = 0.0005   
95% CI [+0.2709, +0.9657]


- What does a full model considering everything say? (Franchise)

In [10]:
X = "ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale_f = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale_f.params['Franchise[T.True]'], full_scale_f.pvalues['Franchise[T.True]']
lo, hi = full_scale_f.conf_int().loc['Franchise[T.True]']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +0.6676  
p-value = 0.0003   
95% CI [+0.3097, +1.0256]


### Result: **Franchise premium survives**

| Model | Franchise effect (ROI) | 95% CI | p-value |
|---|---|---|---|
| Raw (no controls) | *+1.3794* | [+1.0414, +1.7173] | < 0.0001 |
| Content-adjusted, scale not controlled | *+1.1665* | [+0.8329, +1.5001] | < 0.0001 |
| Scale-adjusted only | *+0.6183* | [+0.2709, +0.9657] | 0.0005 |
| Scale + content (full) | *+0.6676* | [+0.3100, +1.0260] | 0.0003 |


1. Though initially Franchise has a +1.38 effect on ROI, about 15% of it is covered via content adjustment and about 55% via Scale-adjustment. Scale is again the bigger absorber, but this time it cannot cover the whole premium.

2. Once both adjustments are done, Franchise films still earn about +0.67 more ROI, and every scale-adjusted CI stays well above zero — unlike H1, the effect does not disappear.

3. The p-values of the scale-adjusted models (0.0005 and 0.0003) are far below 0.05: if franchise status truly did nothing, a gap this size would almost never appear by luck. So the data does **show a definite franchise effect**. 

4. The small rise from +0.62 to +0.67 in the full model is probably because remake films are held fixed there, so franchise is compared against true originals.

#### *Verdict : Franchise premium survives*

### 2. Remakes

- What does RAW comparison say? (Remake)

In [11]:
X = df['Remake']
X = sm.add_constant(X)
rawmodel_r = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel_r.params['Remake'], rawmodel_r.pvalues['Remake']
lo, hi = rawmodel_r.conf_int().loc['Remake']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +0.9555  
p-value = 0.0000   
95% CI [+0.7209, +1.1902]


- What does a content-adjusted (scale not yet controlled) model say? (Remake)

In [12]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel_r = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel_r.params['Remake'], ROImodel_r.pvalues['Remake']
lo, hi = ROImodel_r.conf_int().loc['Remake']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +0.7050  
p-value = 0.0000   
95% CI [+0.4744, +0.9357]


- What does a scale adjustment alone model say? (Remake)

In [13]:
X = "ROI ~ Remake + C(Scale_Bin)"
raw_scale_r = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale_r.params['Remake[T.True]'], raw_scale_r.pvalues['Remake[T.True]']
lo, hi = raw_scale_r.conf_int().loc['Remake[T.True]']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +0.2380  
p-value = 0.0785   
95% CI [-0.0272, +0.5031]


- What does a full model considering everything say? (Remake)

In [14]:
X = "ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale_r = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale_r.params['Remake[T.True]'], full_scale_r.pvalues['Remake[T.True]']
lo, hi = full_scale_r.conf_int().loc['Remake[T.True]']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +0.3012  
p-value = 0.0222   
95% CI [+0.0431, +0.5594]


### Result: **Remake premium is weaker**

| Model | Remake effect (ROI) | 95% CI | p-value |
|---|---|---|---|
| Raw (no controls) | *+0.9555* | [+0.7209, +1.1902] | < 0.0001 |
| Content-adjusted, scale not controlled | *+0.7050* | [+0.4744, +0.9357] | < 0.0001 |
| Scale-adjusted only | *+0.2380* | [-0.0272, +0.5031] | 0.079 |
| Scale + content (full) | *+0.3012* | [+0.0430, +0.5590] | 0.022 |


1. Content barely covers 25% of the raw remake premium , but Scale covers about 75% of the raw remake premium (+0.96 to +0.24).

2. In the scale-only model the CI includes zero (p = 0.079).
    >Implies we cannot conclude remake status on its own can display a clear effect on ROI

3. In the full model other parameters are held fixed, so remakes are compared against true originals
    > The premium rises to +0.30 and becomes more significant (p = 0.022), hence proving the remake premium exists, but it is smaller than the franchise one and is relatively weaker

#### *Verdict : Remake premium survives, but only once franchise films are held out of the comparison*

## H3 : Genre composition at comparable scale
**Do some genres out-earn mainstream films once scale is held fixed?**

### The check: all 13 genres, 4-stage test (raw → content → scale → full)

- For each genre: the same 4-stage ladder as H1/H2; reference category = action films
  (the first genre, dropped during one-hot encoding to avoid the dummy variable trap);
  all p-values are raw, compared against α = 0.05

In [15]:
genres = [c for c in focus_cols if c.startswith('Genre_')]
content_model = sm.OLS(Y, sm.add_constant(df[focus_cols])).fit(cov_type='HC3')
full_model    = smf.ols("ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)", data=df).fit(cov_type='HC3')

def gterm(model, g):
    k = next(k for k in model.params.index if k.split('[')[0] == g)
    return model.params[k], model.pvalues[k]

rows = []
for g in genres:
    m_raw   = sm.OLS(Y, sm.add_constant(df[g])).fit(cov_type='HC3')
    m_scale = smf.ols(f"ROI ~ {g} + C(Scale_Bin)", data=df).fit(cov_type='HC3')
    rc, rp = gterm(m_raw, g);  cc, cp = gterm(content_model, g)
    sc, sp = gterm(m_scale, g); fc, fp_ = gterm(full_model, g)
    rows.append((g.replace('Genre_',''), int(df[g].sum()),
                 f"{rc:+.3f} (p={rp:.3f})", f"{cc:+.3f} (p={cp:.3f})",
                 f"{sc:+.3f} (p={sp:.3f})", f"{fc:+.3f} (p={fp_:.3f})"))

stage = pd.DataFrame(rows, columns=['genre','n','raw','content','scale','full'])
stage = stage.sort_values('full', ascending=False)
print(stage.to_string(index=False))

       genre   n              raw          content            scale             full
   animation   5 -0.593 (p=0.000) -1.069 (p=0.012) -0.320 (p=0.245) -0.479 (p=0.110)
     fantasy  13 -0.535 (p=0.000) -0.406 (p=0.000) -0.677 (p=0.000) -0.436 (p=0.001)
       adult  76 -0.132 (p=0.036) -0.012 (p=0.907) +0.493 (p=0.000) +0.545 (p=0.000)
 documentary   1 +0.871 (p=0.028) +1.377 (p=0.551) -0.026 (p=0.993) +0.478 (p=0.042)
      masala  12 +1.394 (p=0.000) +0.841 (p=0.010) +0.252 (p=0.432) +0.382 (p=0.239)
     rom_com  97 +0.429 (p=0.001) +0.397 (p=0.005) +0.176 (p=0.117) +0.356 (p=0.007)
  love_story 131 +0.029 (p=0.790) +0.090 (p=0.485) -0.017 (p=0.850) +0.177 (p=0.125)
      comedy 282 +0.191 (p=0.014) +0.062 (p=0.557) +0.052 (p=0.444) +0.174 (p=0.075)
       drama 642 -0.262 (p=0.000) -0.050 (p=0.582) -0.029 (p=0.514) +0.164 (p=0.050)
      horror  54 +0.068 (p=0.594) -0.043 (p=0.746) +0.074 (p=0.485) +0.161 (p=0.203)
mythological  14 -0.638 (p=0.000) -0.279 (p=0.022) -0.124 (p=0.16

**Result :** 
- At the full stage (scale + content), four genres pass alpha = 0.05 with a positive premium: **adult** (+0.545, p < 0.001), **rom_com** (+0.356, p = 0.007), **drama** (+0.164, p = 0.050, exactly at the boundary) and **documentary** (+0.478, p = 0.042). Documentary rests on a single film, so we treat it as an anecdote, not a genre effect (see the HC3 discussion in 4.LinearRegression.ipynb); the robust positive premiums are adult and rom_com, with drama borderline. 
- Fantasy is the only significant negative one (-0.436, p = 0.001). 
- Since at least one genre retains significance through all four stages, **H3 is not rejected** 
- The genre with by far the largest premium, adult, is examined in depth below.
*(The log-scale cross-check at the end of this notebook re-tests these premiums on the scale validated in
3.DistributionFitting.ipynb and narrows the surviving genre premium to adult.)*

### Full Scale Check for Adult films

- What does RAW comparison say?

In [16]:
X = df['Genre_adult']
X = sm.add_constant(X)
rawmodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel.params['Genre_adult'], rawmodel.pvalues['Genre_adult']
lo, hi = rawmodel.conf_int().loc['Genre_adult']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : -0.1318  
p-value = 0.0362   
95% CI [-0.2551, -0.0085]


- What does a content-adjusted (scale not yet controlled) model say?

In [17]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel.params['Genre_adult'], ROImodel.pvalues['Genre_adult']
lo, hi = ROImodel.conf_int().loc['Genre_adult']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : -0.0117  
p-value = 0.9074   
95% CI [-0.2083, +0.1849]


- What does a scale adjustment alone model say?

In [18]:
X = "ROI ~ Genre_adult + C(Scale_Bin)"
raw_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale.params['Genre_adult[T.True]'], raw_scale.pvalues['Genre_adult[T.True]']
lo, hi = raw_scale.conf_int().loc['Genre_adult[T.True]']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : +0.4927  
p-value = 0.0000   
95% CI [+0.3546, +0.6308]


- What does a full model considering everything say?

In [19]:
X = "ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale.params['Genre_adult[T.True]'], full_scale.pvalues['Genre_adult[T.True]']
lo, hi = full_scale.conf_int().loc['Genre_adult[T.True]']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : +0.5446  
p-value = 0.0000   
95% CI [+0.3434, +0.7458]


#### Result: **H3 is supported**

| Model | Adult-film effect (ROI) | 95% CI | p-value |
|---|---|---|---|
| Raw (no controls) | *-0.1318* | [-0.2551, -0.0085] | 0.036 |
| Content-adjusted, scale not controlled | *-0.0117* | [-0.2083, +0.1849] | 0.907 |
| Scale-adjusted only | *+0.4927* | [+0.3546, +0.6308] | < 0.0001 |
| Scale + content (full) | *+0.5446* | [+0.3434, +0.7458] | 0.0002 |


1. The raw comparison goes the wrong way: adult films appear to earn significantly **LESS** (-0.13, p = 0.036). But adult films are micro-budget (mean scale 2.30 vs 6.19), so this comparison pits them against blockbusters.

2. Once every film is compared within its own scale band, the sign flips: adult films earn about +0.49 to +0.54 more ROI than comparable mainstream films (p < 0.001) — the strongest and most precisely estimated effect in the project.

3. The 4-stage table above shows the sweep is honest: rom_com (+0.356) and drama (+0.164) also end with significant positive premiums at the full stage, but adult's (+0.545) is by far the largest and most precisely estimated — the contrast motivated by the small-scale ROI dip.

4. In the full model the reference category is action films (dropped during one-hot encoding to avoid the dummy variable trap), and the claim stays scoped: adult films exist only in the small scale bands (76 films, none above scale 5).

#### *Verdict : At comparable scale, the adult genre out-earns mainstream, the negative raw gap was a scale illusion acting against these*

## H4 : Star Power
**Is revenue concentrated among a few recurring stars and directors?**

### 1. Concentration: Gini and top-10 share

**What Gini is:** line up all names poorest → richest; Gini (0 to 1) measures how far that line bends away from perfectly equal revenue — 0 = everyone identical, 1 = one name holds everything.

**Why it is necessary:** the top-10 share depends on an arbitrary cutoff (top-5? top-20?); Gini summarises the whole distribution in one cutoff-free number.

**Here:** names are stars/directors with revenue summed across their films, so "recurring" becomes visible.

- Revenue is summed per name first — concentration is a name-level measure (H4 asks about names, not films)

In [20]:
def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return 2*np.sum(np.arange(1, n+1)*x)/(n*x.sum()) - (n+1)/n

def concentration(names, values):
    tot = pd.DataFrame({'name': names, 'val': values}).groupby('name')['val'].sum()
    return gini(tot.values), tot.nlargest(10).sum()/tot.sum(), tot

star_gini, star_top10, star_tot = concentration(df['Lead Star'], df['Revenue'])
dir_gini, dir_top10, dir_tot = concentration(df['Director'], df['Revenue'])

print(f"Stars    : {len(star_tot)} names | Gini = {star_gini:.3f} | top-10 share = {100*star_top10:.1f}%")
print(f"Directors: {len(dir_tot)} names | Gini = {dir_gini:.3f} | top-10 share = {100*dir_top10:.1f}%")
print("\nTop-10 stars by total revenue:")
for nm, v in star_tot.nlargest(10).items():
    print(f"  {nm:25s} {v/1e9:6.2f} B  ({100*v/star_tot.sum():.1f}%)")

Stars    : 761 names | Gini = 0.954 | top-10 share = 59.3%
Directors: 1045 names | Gini = 0.897 | top-10 share = 25.6%

Top-10 stars by total revenue:
  Salman Khan                46.13 B  (11.4%)
  Akshay Kumar               41.05 B  (10.2%)
  Shahrukh Khan              31.90 B  (7.9%)
  Aamir Khan                 29.03 B  (7.2%)
  Ajay Devgn                 24.39 B  (6.0%)
  Ranbir Kapoor              17.22 B  (4.3%)
  Hrithik Roshan             16.95 B  (4.2%)
  Saif Ali Khan              12.05 B  (3.0%)
  Ranveer Singh              11.08 B  (2.7%)
  Prabhas                     9.70 B  (2.4%)


### 2. Scale-managed star ranking: does star identity matter beyond scale?

- Gini tells us the revenue is concentrated by the top 10%, but it never tells us if this is because of the movies themselves or the stars.
> - If the films themselves are very unequal (a few blockbusters hold most of the revenue), then whoever is tagged to those films *looks* like a star. A high Gini can therefore be false concentration: scale, not stardom.
> - We seek to remove such cases here
- **ROI ~ C(star) + C(Scale_Bin):** manage scale first, then rank stars by their scale-adjusted premium; stars with at least 5 films only (a 1-film star ranks by pure residual luck)

In [21]:
df['star'] = df['Lead Star']
m_scale = smf.ols("ROI ~ C(Scale_Bin)", data=df).fit()
m_star  = smf.ols("ROI ~ C(star) + C(Scale_Bin)", data=df).fit()
print(f"R2 with scale bands only : {m_scale.rsquared:.3f}")
print(f"R2 with scale + star name: {m_star.rsquared:.3f}   (stars add {m_star.rsquared-m_scale.rsquared:+.3f})")

vc = df['star'].value_counts()
rows = []
for k in m_star.params.index:
    if k.startswith('C(star)'):
        star = k.split('[')[1].rstrip(']').strip("'T.")
        rows.append((star, int(vc.get(star, 0)), m_star.params[k]))
rank = pd.DataFrame(rows, columns=['star', 'n_films', 'premium'])
rank = rank[rank.n_films >= 5].sort_values('premium', ascending=False)
print(f"\nTop 10 stars by scale-adjusted premium (>=5 films required, {len(rank)} eligible):")
print(rank.head(10).round(3).to_string(index=False))

R2 with scale bands only : 0.332
R2 with scale + star name: 0.540   (stars add +0.207)



Top 10 stars by scale-adjusted premium (>=5 films required, 72 eligible):
             star  n_films  premium
       Aamir Khan       10    3.724
     Varun Dhawan        7    2.833
    Shahrukh Khan       16    2.616
      Vidya Balan        7    2.498
       Imran Khan        9    2.423
Sidharth Malhotra        5    2.372
 Ayushman Khurana        8    2.300
      Salman Khan       26    2.160
     Arjun Kapoor        6    2.119
   Hrithik Roshan       11    2.097


#### Result: **H4 is supported**

| | Stars | Directors |
|---|---|---|
| Names | 761 | 1045 |
| Gini | *0.954* | *0.897* |
| Top-10 share (observed) | *59.3%* | *25.6%* |
| Top-10 share under perfect equality | 1.3% | 1.0% |


1. The concentration itself is extreme: the top-10 stars hold 59.3% of all revenue against just 1.3% under perfect equality (10/761), and the Gini of 0.954 confirms near-monopoly.

2. The scale-managed check: explaining ROI with scale bands alone gives R2 = 0.332; adding star names lifts it to 0.540 — star identity adds +0.21 of explained variance beyond film size.

3. The scale-adjusted ranking (stars with at least 5 films) puts Aamir Khan (+3.72), Varun Dhawan (+2.83) and Shahrukh Khan (+2.62) at the top — familiar names, but now ranked by what they earn beyond what their film sizes alone would earn.

4. Caveats: the ranking uses point estimates only (a 760-dummy model has no valid HC3 errors), a single mega-hit can lift a star's average, directors are far less concentrated than stars (25.6% vs 59.3%), and this is association, not causation — big films make stars as much as stars make big films.

#### *Verdict : Revenue concentration is real and extreme, and star identity keeps explaining returns even after film scale is accounted for*

## Cross-check: every verdict on the log scale (per 3.DistributionFitting.ipynb)
**Notebook 3's distribution fitting — the project's validated 'best' scale — showed ROI is near-normal on the log scale (lognormal wins the χ² contest; skew −0.36 vs +2.23 raw). Every verdict above was drawn on raw ROI, so we re-run the same ladders with `log(ROI)` and HC3 errors. Coefficients are approximately percentage effects (×100); the verdicts should agree if they were not artifacts of the raw scale.

In [22]:
df['log_ROI'] = np.log(df['ROI'])

def ladder4_log(flag):
    """raw -> content -> scale -> full ladder for one flag, response = log(ROI), HC3."""
    m_raw   = sm.OLS(df['log_ROI'], sm.add_constant(df[flag].astype(int))).fit(cov_type='HC3')
    m_cont  = sm.OLS(df['log_ROI'], sm.add_constant(df[focus_cols])).fit(cov_type='HC3')
    m_scale = smf.ols(f"log_ROI ~ {flag} + C(Scale_Bin)", data=df).fit(cov_type='HC3')
    m_full  = smf.ols("log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)", data=df).fit(cov_type='HC3')
    rows = []
    for stage, m, key in [('raw', m_raw, flag), ('content', m_cont, flag),
                          ('scale', m_scale, f'{flag}[T.True]'), ('full', m_full, f'{flag}[T.True]')]:
        k = next(k for k in m.params.index if k.split('[')[0] == flag)
        rows.append((stage, m.params[k], m.pvalues[k]))
    return rows

for flag in ['Holiday', 'Franchise', 'Remake']:
    print(f"=== {flag} on log(ROI) (HC3) ===")
    for stage, coef, pv in ladder4_log(flag):
        print(f"  {stage:8s} {coef:+.4f}  (p = {pv:.4f})")
    print()

log_full = smf.ols("log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)", data=df).fit(cov_type='HC3')
rows = []
for g in [c for c in focus_cols if c.startswith('Genre_')]:
    k = next(k for k in log_full.params.index if k.split('[')[0] == g)
    rows.append((g.replace('Genre_', ''), int(df[g].sum()), log_full.params[k], log_full.pvalues[k]))
genre_log = pd.DataFrame(rows, columns=['genre', 'n', 'coef', 'p']).sort_values('coef', ascending=False)
print("=== Genres on log(ROI), full stage (HC3) ===")
print(genre_log.round(4).to_string(index=False))

=== Holiday on log(ROI) (HC3) ===
  raw      +0.2786  (p = 0.0007)
  content  +0.1057  (p = 0.1321)
  scale    +0.0184  (p = 0.7641)
  full     -0.0153  (p = 0.7927)

=== Franchise on log(ROI) (HC3) ===
  raw      +1.7019  (p = 0.0000)
  content  +1.1722  (p = 0.0000)
  scale    +0.3794  (p = 0.0001)
  full     +0.3356  (p = 0.0006)

=== Remake on log(ROI) (HC3) ===


  raw      +1.6008  (p = 0.0000)
  content  +1.0892  (p = 0.0000)
  scale    +0.3397  (p = 0.0002)
  full     +0.3488  (p = 0.0001)

=== Genres on log(ROI), full stage (HC3) ===
       genre   n    coef      p
       adult  76  1.6236 0.0000
 documentary   1  0.8236 0.8370
      horror  54  0.4384 0.0126
     rom_com  97  0.1892 0.2169
      masala  12  0.1875 0.3288
mythological  14  0.0946 0.7719
    thriller 216  0.0593 0.6021
       drama 642  0.0258 0.8027
      comedy 282 -0.0010 0.9928
  love_story 131 -0.1443 0.2940
    suspense  29 -0.2122 0.3976
   animation   5 -0.8917 0.2283
     fantasy  13 -1.0864 0.0000


**Result — every verdict is now tested on the scale 3.DistributionFitting.ipynb validated.**
- **H1 (Holiday): rejected on both scales.** Full model −0.02 (p = 0.79); the whole raw premium (+0.28) is
  absorbed by content and scale controls.
- **H2 survives.** Franchise +0.34 ≈ **+40% ROI** (p = 0.0006), Remake +0.35 ≈ **+42%** (p = 0.0001). The
  ladders shrink +1.70 → +0.34 and +1.60 → +0.35: scale absorbs ~80% of the raw premium and the remainder is
  real — the same shape as the raw-ROI ladders above.
- **H3 narrows to adult.** adult +1.62 ≈ **+407% ROI** (p < 0.0001) and fantasy's penalty (−1.09, p < 0.0001)
  are confirmed, and horror emerges (+0.44, p = 0.013); but **rom_com (p = 0.22) and drama (p = 0.80) —
  significant on raw ROI — do not survive the log scale**, so H3's claim is scoped to the adult genre.
- **H4** needs no re-check: the Gini and the scale-binned star ranking are distribution-free comparisons of
  revenue shares and within-band ROI, not distribution-dependent models.

The verdicts of this notebook therefore hold on both the raw and the validated log scale, with H3 honestly
narrowed to the adult genre.